# D. Morphology 

This notebook uses the previously created object and progress csv files, to create small image cutouts and masks, then run statmorph anaylsis. Also Appends the Key CSV with statmorph morphology results.

User can itterate through the image tract/patch combination, n, and the object within that image, m, sorted by ascending tract, patch value  

Install statmorph if you have not

In [ ]:
#pip install statmorph

In [ ]:
#python -c "import statmorph.tests; statmorph.tests.runall()"

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import getpass 
import matplotlib.pyplot as plt
import astropy.units as u
import statmorph

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from photutils.segmentation import detect_sources
from astropy.stats import sigma_clipped_stats
from statmorph.utils.image_diagnostics import make_figure

In [ ]:
my_username = getpass.getuser()

## IF YOU CHANGE/ WORKING UNDER DIFFERENT FOLDER YOU MUST CHANGE THIS , KB_462 IS MINE
BASE = Path(f"/home/{my_username}/KB_462")
DATA = BASE / "data"

FITS_PATH = DATA / "fits_files"
PLOTS_PATH = DATA / "plots" / "morphology"

PLOTS_PATH.mkdir(parents=True, exist_ok=True)

print("DATA:", DATA)
print("FITS:", FITS_PATH)

## Part 1 

# Input

Input the progress csv and select the n value for the desired tract/patch combination

The bold number on the left is the n value for next section

In [ ]:
#
Input1 = DATA / "Tract_Patch_Visit_Name.csv"
#

df1 = pd.read_csv(Input1)

MM = len(df1)
tract = df1["tract"].to_numpy()
patch = df1["patch"].to_numpy()
numVis = df1["NumVisIm"].to_numpy()
File_Path = df1["Name"].to_numpy()
numObjs = df1["NumObjects"].to_numpy()
df1

# Select Tract and Patch Combination

Select an individual tract patch combination 

In [ ]:
## USER INPUT:

In [ ]:
## SELECT THE N VALUE OF THE DESIRED TRACT, PATCH COMBINATION
n = 0
#

This is the number of objects in the image we have.

In [ ]:
tract[n]
patch[n]
File_Path[n]
numObjs[n]
M = int(numObjs[n])

print("Coadd number ",(n+1), " Out of ",MM)
print("Objects in image: ",M)


# Quick Check

Verify that the specific file in in directory 

In [ ]:
my_username = getpass.getuser()
os.listdir('/home//'+my_username)

In [ ]:
print(File_Path)

If the previous step fails you can import the .fits file from the directory manually

In [ ]:
## INPUT FILE PATH FOR CURRENT .FITS FILE

file_name = Path(File_Path[n].replace("file://", "")).name
file_path = FITS_PATH / file_name

# MANUAL INPUT

#file_path = "/home/kibuckin/deep_coadd_predetection_5063_45_r_lsst_cells_v1_u_kibuckin_custom_coadd_ecdfs_20260331T035152Z.fits"
#file_path = "/home/kibuckin/deep_coadd_predetection_2394_46_r_lsst_cells_v1_u_kibuckin_custom_coadd_ecdfs_20260331T205726Z.fits"
#file_path = "/home/user/"
#file_path = "/home/kibuckin/deep_coadd_predetection_2234_85_r_lsst_cells_v1_u_kibuckin_custom_coadd_ecdfs_20260408T062950Z.fits"


hdul = fits.open(file_path)
hdul.info()
image = hdul[1].data # 1 = image
wcs = WCS(hdul[1].header)  
variance = hdul[3].data

In [ ]:
hdul[1]

# Part 2

Select specific object within that image 

# Input 

Input the sorted "Key Objects" list

In [ ]:
#
Input = DATA/  "Key_A.csv"
#

df = pd.read_csv(Input)
df2 = pd.read_csv(Input)

tp = df2[['tract','patch']].drop_duplicates().to_numpy()
my_tract,my_patch = tp[n]

# Select Object Within Tract and Patch

use the 'm' value to itterate through each object of interest from the key objects csv. if there is 1 object then m=0 and only 0. if there is 5 objects then m can be m=0, m=1, m=2, m=3, m=4. i would recommend proceeding the that manner for images with multiple objects

In [ ]:
## USER INPUT:

In [ ]:
## SELECT OBJECT WITHIN THE IMAGE
m = 0
#

tp2 = df2[(df2['tract'] == my_tract) & (df2['patch'] == my_patch)].reset_index(drop=True)
row = tp2.iloc[m]


print("Tract ", my_tract ,"Patch ", my_patch)
print("Object ", (m+1) ," Out of ", M)

Loads image/row data

In [ ]:
#Load row data
ObjID = int(row["LSST_objID"])
memtag = row["member_tag"]
ra = row["RA"]
dec = row["DEC"]
Z = row["Z"]
sersic_index = row["sersic_index"]
TRACT = row["tract"]
PATCH = row["patch"]
sersic_x = row["sersic_x"]
sersic_y = row["sersic_y"]
sersic_reff_x = row["sersic_reff_x"]
sersic_reff_y = row["sersic_reff_y"]
sersic_rho = row["sersic_rho"]
s_ra = row["sersic_ra"]
s_dec = row["sersic_dec"]
uFlux_LSST = row["uFlux_LSST"]
gFlux_LSST = row["gFlux_LSST"]
rFlux_LSST = row["rFlux_LSST"]
iFlux_LSST = row["iFlux_LSST"]
zFlux_LSST = row["zFlux_LSST"]
yFlux_LSST = row["yFlux_LSST"]
uFluxErr_LSST = row["uFluxErr_LSST"]
gFluxErr_LSST = row["gFluxErr_LSST"]
rFluxErr_LSST = row["rFluxErr_LSST"]
iFluxErr_LSST = row["iFluxErr_LSST"]
zFluxErr_LSST = row["zFluxErr_LSST"]
yFluxErr_LSST = row["yFluxErr_LSST"]
visFlux_EUCL_1aper = row["visFlux_EUCL_1aper"]
visFlux_EUCL_2aper = row["visFlux_EUCL_2aper"]
visFlux_EUCL_3aper = row["visFlux_EUCL_3aper"]
yFlux_EUCL_1aper = row["yFlux_EUCL_1aper"]
yFlux_EUCL_2aper = row["yFlux_EUCL_2aper"]
yFlux_EUCL_3aper = row["yFlux_EUCL_3aper"]
jFlux_EUCL_1aper = row["jFlux_EUCL_1aper"]
jFlux_EUCL_2aper = row["jFlux_EUCL_2aper"]
jFlux_EUCL_3aper = row["jFlux_EUCL_3aper"]
hFlux_EUCL_1aper = row["hFlux_EUCL_1aper"]
hFlux_EUCL_2aper = row["hFlux_EUCL_2aper"]
hFlux_EUCL_3aper = row["hFlux_EUCL_3aper"]
visFlux_sersic = row["visFlux_sersic"]
yFlux_sersic = row["yFlux_sersic"]
jFlux_sersic = row["yFlux_sersic"]
hFlux_sersic = row["yFlux_sersic"]
visFluxErr_EUCL_1aper = row["visFluxErr_EUCL_1aper"]
visFluxErr_EUCL_2aper = row["visFluxErr_EUCL_2aper"]
visFluxErr_EUCL_3aper = row["visFluxErr_EUCL_3aper"]
yFluxErr_EUCL_1aper = row["yFluxErr_EUCL_1aper"]
yFluxErr_EUCL_2aper = row["yFluxErr_EUCL_2aper"]
yFluxErr_EUCL_3aper = row["yFluxErr_EUCL_3aper"]
jFluxErr_EUCL_1aper = row["jFluxErr_EUCL_1aper"]
jFluxErr_EUCL_2aper = row["jFluxErr_EUCL_2aper"]
jFluxErr_EUCL_3aper = row["jFluxErr_EUCL_3aper"]
hFluxErr_EUCL_1aper = row["hFluxErr_EUCL_1aper"]
hFluxErr_EUCL_2aper = row["hFluxErr_EUCL_2aper"]
hFluxErr_EUCL_3aper = row["hFluxErr_EUCL_3aper"]
visFluxErr_sersic = row["visFluxErr_sersic"]
yFluxErr_sersic = row["yFluxErr_sersic"]
jFluxErr_sersic = row["yFluxErr_sersic"]
hFluxErr_sersic = row["yFluxErr_sersic"]



Cra = (ra + s_ra) /2
Cdec = (dec + s_dec) /2
difRa = (ra - s_ra)
difDec = (dec  - s_dec)


print(ra,dec)
print(Cra,Cdec)
print(s_ra,s_dec)
print(difRa,difDec)
#df

# View Coadd

Convert ra/dec to local pixel coordinate. limit is 3400

In [ ]:
x, y = wcs.world_to_pixel_values(ra,dec)
print("Object pixel coordinates:",x,",", y)

quick view a custom made deep coadd, with object location

Expected Error: RuntimeWarning: invalid value encountered in log10

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(np.log10(image + 1), origin='lower', cmap='gray')
plt.colorbar()
plt.title("Deep Coadd Quick View")

plt.scatter(x, y, color='red', s=25)

plt.show()

# Create Cutout and Mask

Specific pixel cutout view. User might need to adjust central pixel coordinate by a few pixel to center on the object

In [ ]:
## POTENTIAL USER INPUT:

In [ ]:
# CREATE CUTOUT
x = np.round(x)
y = np.round(y)

###
# PIXEL LOCATION MIGHT NEED TO BE SLIGHT ADJUSTED TO CENTER ON OBJECT MASK
x_0 = x - -2
y_0 = y - -1
# REGION SIZE MIGHT NEED TO BE ADJUSTED
region = 10
###

xmin = int(x_0 - region )
xmax = int(x_0 + region)
ymin = int(y_0 - region )
ymax = int(y_0 + region )

frame = image[ymin:ymax, xmin:xmax]

Loc = (x_0, y_0) 
size = region * 2
Dim = (size, size)

cutout = Cutout2D(image, Loc, Dim)
cutout_var = Cutout2D(variance, Loc, Dim, wcs=wcs)
weightmap = 1 / cutout_var.data
cutout_data = cutout.data

Mask sensitivity might need to be adjusted to get good cutout. Range is typically 1.30 to 3.0

If you get NoDetectionsWarning you should lower the threshold

In [ ]:
## POTENTIAL USER INPUT:

In [ ]:
# CREATE MASKS
mean, median, std = sigma_clipped_stats(cutout_data)

# THRESHOLD MIGHT NEED TO BE ADJUSTED, GOOD VALUE IS 1.33 OR 1.35
threshold = median + 1.0*std
#

#threshold = median + 1.30*std
#threshold = median + 1.35*std
#threshold = median + 1.40*std
#threshold = median + 1.50*std
#threshold = median + 2.0*std
#threshold = median + 3.0*std



segm = detect_sources(cutout_data, threshold, npixels=5)
print(segm)
mask = segm.data > 0

# Save, Display  Cutout and Mask 

In [ ]:
CUTOUT_PATH = DATA / "plots" / "morphology" / str(ObjID)
CUTOUT_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow((frame), origin='lower', cmap='gray')

plt.colorbar()
plt.title(f"{ObjID} Cutout and Mask")
plt.contour(mask, colors='red')
plt.scatter(region, region, color='red', s=1)

plt.savefig(CUTOUT_PATH / "cutout_mask.png",
            dpi=300, bbox_inches="tight")

plt.show()
plt.close()

# Quick Check

If this output = 0 , then you are not centered on a mask. you need to either A) adjust pixel coordinates, region to be centered on the object . or B) Adjust the mask sensitivity threshold 

In [ ]:
yc, xc = cutout.data.shape[0]//2, cutout.data.shape[1]//2

label = segm.data[yc, xc]
print("Center label:", label)

In [ ]:
#A
## Changeing A might look like 
###
# PIXEL LOCATION MIGHT NEED TO BE SLIGHT ADJUSTED TO CENTER ON OBJECT MASK
#x_0 = x - 2
#y_0 = y - 1
# REGION SIZE MIGHT NEED TO BE ADJUSTED
#region = 8
###

In [ ]:
#B
## Changeing B might look like 
###
# THRESHOLD MIGHT NEED TO BE ADJUSTED, GOOD VALUE IS 1.33 OR 1.35
#threshold = median + 2.0*std
###

## Part 3

# Morphology

In [ ]:
morph = statmorph.source_morphology(
    cutout.data,
    mask.astype(int),
    weightmap=weightmap
)

In [ ]:
morph = morph[0]

In [ ]:
print('BASIC MEASUREMENTS (NON-PARAMETRIC)')
print('xc_centroid =', morph.xc_centroid)
print('yc_centroid =', morph.yc_centroid)
print('ellipticity_centroid =', morph.ellipticity_centroid)
print('elongation_centroid =', morph.elongation_centroid)
print('orientation_centroid =', morph.orientation_centroid)
print('xc_asymmetry =', morph.xc_asymmetry)
print('yc_asymmetry =', morph.yc_asymmetry)
print('ellipticity_asymmetry =', morph.ellipticity_asymmetry)
print('elongation_asymmetry =', morph.elongation_asymmetry)
print('orientation_asymmetry =', morph.orientation_asymmetry)
print('rpetro_circ =', morph.rpetro_circ)
print('rpetro_ellip =', morph.rpetro_ellip)
print('rhalf_circ =', morph.rhalf_circ)
print('rhalf_ellip =', morph.rhalf_ellip)
print('r20 =', morph.r20)
print('r80 =', morph.r80)
print('Gini =', morph.gini)
print('M20 =', morph.m20)
print('F(G, M20) =', morph.gini_m20_bulge)
print('S(G, M20) =', morph.gini_m20_merger)
print('sn_per_pixel =', morph.sn_per_pixel)
print('C =', morph.concentration)
print('A =', morph.asymmetry)
print('S =', morph.smoothness)
print()
print('SERSIC MODEL')
print('sersic_amplitude =', morph.sersic_amplitude)
print('sersic_rhalf =', morph.sersic_rhalf)
print('sersic_n =', morph.sersic_n)
print('sersic_xc =', morph.sersic_xc)
print('sersic_yc =', morph.sersic_yc)
print('sersic_ellip =', morph.sersic_ellip)
print('sersic_theta =', morph.sersic_theta)
print('sersic_chi2_dof =', morph.sersic_chi2_dof)
print()
print('OTHER')
print('sky_mean =', morph.sky_mean)
print('sky_median =', morph.sky_median)
print('sky_sigma =', morph.sky_sigma)
print('flag =', morph.flag)
print('flag_sersic =', morph.flag_sersic)

working statmorph baby

In [ ]:
OBJ_PATH = DATA / "plots" / "morphology" / str(ObjID)
OBJ_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
fig = make_figure(morph)
fig.savefig(OBJ_PATH / "statmorph_output.png",dpi=300, bbox_inches="tight")

# Update New CSV

In [ ]:
##
# CREATE A NEW CSV FILE TO TRACK PROGRESS AS WE PROCESS IMAGES
##


#OPTION TO CHANGE TO NEW FILE BY CHANGING THIS NAME
filename = DATA/ "ObjID_Statmorph1.csv"

row_df = pd.DataFrame([{
        "Object ID": ObjID,
        "member_tag":memtag,
        "RA": ra,
        "DEC": dec,
        "Z" : Z,
        "sersic_index":sersic_index ,
        "tract": TRACT ,
        "patch": PATCH ,
        "sersic_x": sersic_x ,
        "sersic_y":sersic_y,
        "sersic_reff_x":sersic_reff_x ,
        "sersic_reff_y":sersic_reff_y ,
        "sersic_rho":sersic_rho ,
        "sersic_ra": s_ra ,
        "sersic_dec":s_dec ,
        "uFlux_LSST":uFlux_LSST ,
        "gFlux_LSST": gFlux_LSST ,
        "rFlux_LSST":rFlux_LSST ,
        "iFlux_LSST":iFlux_LSST ,
        "zFlux_LSST":zFlux_LSST ,
        "yFlux_LSST":yFlux_LSST ,
        "uFluxErr_LSST":uFluxErr_LSST ,
        "gFluxErr_LSST":gFluxErr_LSST ,
        "rFluxErr_LSST":rFluxErr_LSST ,
        "iFluxErr_LSST":iFluxErr_LSST ,
        "zFluxErr_LSST":zFluxErr_LSST ,
        "yFluxErr_LSST":yFluxErr_LSST ,
        "visFlux_EUCL_1aper":visFlux_EUCL_1aper ,
        "visFlux_EUCL_2aper":visFlux_EUCL_2aper ,
        "visFlux_EUCL_3aper":visFlux_EUCL_3aper ,
        "yFlux_EUCL_1aper":yFlux_EUCL_1aper ,
        "yFlux_EUCL_2aper":yFlux_EUCL_2aper ,
        "yFlux_EUCL_3aper":yFlux_EUCL_3aper ,
        "jFlux_EUCL_1aper":jFlux_EUCL_1aper ,
        "jFlux_EUCL_2aper":jFlux_EUCL_2aper ,
        "jFlux_EUCL_3aper":jFlux_EUCL_3aper ,
        "hFlux_EUCL_1aper":hFlux_EUCL_1aper ,
        "hFlux_EUCL_2aper":hFlux_EUCL_2aper ,
        "hFlux_EUCL_3aper":hFlux_EUCL_3aper ,
        "visFlux_sersic":visFlux_sersic ,
        "yFlux_sersic":yFlux_sersic ,
        "jFlux_sersic":jFlux_sersic ,
        "hFlux_sersic":hFlux_sersic ,
        "visFluxErr_EUCL_1aper":visFluxErr_EUCL_1aper ,
        "visFluxErr_EUCL_2aper":visFluxErr_EUCL_2aper ,
        "visFluxErr_EUCL_3aper":visFluxErr_EUCL_3aper ,
        "yFluxErr_EUCL_1aper":yFluxErr_EUCL_1aper ,
        "yFluxErr_EUCL_2aper":yFluxErr_EUCL_2aper ,
        "yFluxErr_EUCL_3aper":yFluxErr_EUCL_3aper ,
        "jFluxErr_EUCL_1aper":jFluxErr_EUCL_1aper ,
        "jFluxErr_EUCL_2aper":jFluxErr_EUCL_2aper ,
        "jFluxErr_EUCL_3aper":jFluxErr_EUCL_3aper ,
        "hFluxErr_EUCL_1aper":hFluxErr_EUCL_1aper ,
        "hFluxErr_EUCL_2aper":hFluxErr_EUCL_2aper ,
        "hFluxErr_EUCL_3aper":hFluxErr_EUCL_3aper ,
        "visFluxErr_sersic":visFluxErr_sersic ,
        "yFluxErr_sersic":yFluxErr_sersic ,
        "jFluxErr_sersic":jFluxErr_sersic ,
        "hFluxErr_sersic":hFluxErr_sersic ,
        #
        "xc_centroid":morph.xc_centroid,
        "yc_centroid":morph.yc_centroid,
        "ellipticity_centroid":morph.ellipticity_centroid,
        "elongation_centroid":morph.elongation_centroid,
        "orientation_centroid":morph.orientation_centroid,
        "xc_asymmetry":morph.xc_asymmetry,
        "yc_asymmetry":morph.yc_asymmetry,
        "ellipticity_asymmetry":morph.ellipticity_asymmetry,
        "elongation_asymmetry":morph.elongation_asymmetry,
        "orientation_asymmetry":morph.orientation_asymmetry,
        "rpetro_circ":morph.rpetro_circ,
        "rpetro_ellip":morph.rpetro_ellip,
        "rhalf_circ":morph.rhalf_circ,
        "rhalf_ellip":morph.rhalf_ellip,
        "r20":morph.r20,
        "r80":morph.r80,
        "gini":morph.gini,
        "m20":morph.m20,
        "gini_m20_bulge":morph.gini_m20_bulge,
        "gini_m20_merger":morph.gini_m20_merger,
        "sn_per_pixel":morph.sn_per_pixel,
        "concentration":morph.concentration,
        "asymmetry":morph.asymmetry,
        "smoothness":morph.smoothness,
        "sersic_amplitude":morph.sersic_amplitude,
        "sersic_rhalf":morph.sersic_rhalf,
        "sersic_n":morph.sersic_n,
        "sersic_xc": morph.sersic_xc,
        "sersic_yc":morph.sersic_yc,
        "sersic_ellip":morph.sersic_ellip,
        "sersic_theta":morph.sersic_theta,
        "sersic_chi2_dof": morph.sersic_chi2_dof,
        "sky_mean":morph.sky_mean,
        "sky_median":morph.sky_median,
        "sky_sigma":morph.sky_sigma,
        "flag":morph.flag,
        "flag_sersic":morph.flag_sersic
        
}])

row_df.to_csv(filename, mode='a', header=not filename.exists(), index=False)